# LangChain Agent vs. Mellea Agent: Planning a Trip, Two Ways

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/LangChainAgentVsMelleaAgent/LangChainAgentVsMelleaAgent.ipynb)

An **agent** lets a model decide, turn by turn, which tool to call and when to stop. This recipe builds the same vacation-planning agent twice, sharing one tool:

- [LangChain](https://www.langchain.com/)'s `create_agent`, validated with a hand-rolled retry.
- [Mellea](https://mellea.ai)'s `react()`, validated with a `Requirement` on a `@generative` function.
- Both use the **ReACT** (Reason + Act) pattern. The model reasons, calls a tool, reads the result, and repeats until it can answer.
- Both return the same typed `VacationPlan`, checked against the same budget.

**What do the two differ in?** → LangChain decides what the agent does next. Mellea decides whether each output is good enough to use.

Runs on one local Ollama model. For watsonx.ai, see [Quick Start](../QuickStart/QuickStart.ipynb).

## Step 1. Set up your environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it and run locally.

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

## Step 2. Install relevant libraries

We need `mellea` for the Mellea agent, `langchain` and `langchain-ollama` for the LangChain agent, and `pydantic` for the typed plan.

In [ ]:
%pip install -q git+https://github.com/ibm-granite-community/utils
%pip install -q mellea langchain langchain-ollama pydantic

## Step 3. Set up a local Ollama backend

Both agents use the same local Ollama model. Skip this cell if Ollama is already running with the model pulled.

*Mellea guide: [Ollama: Install Ollama](https://github.com/generative-computing/mellea/blob/main/docs/docs/integrations/ollama.md#install-ollama)*

In [ ]:
# Install Ollama
! sudo apt-get install -y zstd
! curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama daemon in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Pull the IBM Granite model used by both agents below
! ollama pull granite4.2:3b

## Step 4. Import libraries and set constants

- `OLLAMA_MODEL_TAG` is shared, so both agents use the same model.
- `TOOL_CALL_BUDGET` caps the agent loop for both.
- `MAX_NEW_TOKENS` caps output length for both. Mellea's `react()` writes its whole answer into one `final_answer` tool call, and a 1024 cap sometimes cut it off.
- `TRIP_DAYS`, `DAILY_BUDGET_USD` and `vacation_request` describe the trip both agents plan.

In [ ]:
import json

import pydantic
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

from mellea import MelleaSession, generative, start_session
from mellea.backends.model_options import ModelOption
from mellea.backends.tools import MelleaTool
from mellea.core import Requirement, ValidationResult
from mellea.stdlib.context import ChatContext
from mellea.stdlib.frameworks.react import react
from mellea.stdlib.sampling import RepairTemplateStrategy

OLLAMA_MODEL_TAG = "granite4.2:3b"   # shared by ChatOllama (LangChain) and the Mellea session
TOOL_CALL_BUDGET = 8                 # loop control cap, applied to both agents below
MAX_NEW_TOKENS = 2048                # output-length cap, both agents: bounds a runaway generation

TRIP_DAYS = 4
DAILY_BUDGET_USD = 120

vacation_request = (
    f"Plan a {TRIP_DAYS}-day vacation for a traveler with a budget of "
    f"${DAILY_BUDGET_USD} per day who wants warm weather, loves nature and local "
    "culture, and is traveling in April. Recommend one destination and give a "
    "day-by-day itinerary with an estimated total cost."
)

print("Model:", OLLAMA_MODEL_TAG)
print("Request:", vacation_request)

## Step 5. One shared tool for both agents

- `@tool` turns a plain function into a LangChain `BaseTool`, using its name, docstring and type hints to describe it to the model.
- `destination_guide` looks up cost, best months, climate and highlights for six destinations. It runs locally, with no API key or network call.
- The model only *asks* for a lookup; the agent loop runs the function.

Step 9 hands this same tool to Mellea unmodified.

*Note:* `destination_guide` returns JSON text, not a dict. When a tool result contains plain numbers, Mellea's formatter logs `formatter encountered an unexpected type in _stringify; using str() on <class 'int'>`.

*Mellea guide: [Tools and Agents: Defining tools with `@tool`](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#defining-tools-with-tool) (Mellea's own `@tool` works the same way)*

In [ ]:
DESTINATIONS = {
    "Bali, Indonesia": {
        "avg_daily_cost_usd": 65,
        "best_months": ["Apr", "May", "Jun", "Sep"],
        "climate": "tropical",
        "highlights": ["beaches", "rice terraces", "temples"],
    },
    "Costa Rica": {
        "avg_daily_cost_usd": 95,
        "best_months": ["Dec", "Jan", "Feb", "Mar"],
        "climate": "tropical",
        "highlights": ["rainforest hikes", "wildlife", "beaches"],
    },
    "Marrakech, Morocco": {
        "avg_daily_cost_usd": 70,
        "best_months": ["Mar", "Apr", "Oct", "Nov"],
        "climate": "arid",
        "highlights": ["markets", "desert day trips", "architecture"],
    },
    "Reykjavik, Iceland": {
        "avg_daily_cost_usd": 210,
        "best_months": ["Jun", "Jul", "Aug"],
        "climate": "subarctic",
        "highlights": ["waterfalls", "hot springs", "northern lights"],
    },
    "Tokyo, Japan": {
        "avg_daily_cost_usd": 150,
        "best_months": ["Mar", "Apr", "Oct", "Nov"],
        "climate": "temperate",
        "highlights": ["temples", "food markets", "gardens"],
    },
    "Lisbon, Portugal": {
        "avg_daily_cost_usd": 100,
        "best_months": ["Apr", "May", "Jun", "Sep"],
        "climate": "temperate",
        "highlights": ["old town", "coastal day trips", "local cuisine"],
    },
}


@tool
def destination_guide(destination: str) -> str:
    """Look up travel-planning facts for a destination.

    Args:
        destination: Destination name, e.g. "Lisbon, Portugal".
    """
    search_term = destination.strip().lower()
    for name, facts in DESTINATIONS.items():
        if search_term in name.lower() or name.lower() in search_term:
            return json.dumps({"destination": name, **facts})
    return json.dumps({"error": f"No data for '{destination}'.", "known_destinations": list(DESTINATIONS)})


lisbon_facts = destination_guide.invoke({"destination": "Lisbon"})
print("Lookup for 'Lisbon' found:", lisbon_facts)

## Step 6. A typed itinerary schema, and a validator for it

- Both agents return a `VacationPlan` instead of free text.
- A schema guarantees *shape*, not correct numbers. `budget_check` checks the day count and total cost in plain Python.
- `run_with_validated_retry` is LangChain's hand-rolled retry (Step 8). It runs the agent, checks the plan, and tries again on failure.

In [ ]:
class DayPlan(pydantic.BaseModel):
    day: int = pydantic.Field(description="Day number, starting at 1")
    activities: list[str] = pydantic.Field(description="Planned activities for this day")


class VacationPlan(pydantic.BaseModel):
    destination: str = pydantic.Field(description="The recommended destination")
    itinerary: list[DayPlan] = pydantic.Field(description="One entry per day of the trip")
    estimated_total_cost_usd: float = pydantic.Field(description="Estimated total cost in USD for the whole trip")


def budget_check(plan: VacationPlan) -> tuple[bool, str]:
    """Deterministic check: the itinerary covers every day and fits the budget."""
    if len(plan.itinerary) != TRIP_DAYS:
        return False, f"Itinerary has {len(plan.itinerary)} days; expected {TRIP_DAYS}."
    max_cost = DAILY_BUDGET_USD * TRIP_DAYS
    if plan.estimated_total_cost_usd > max_cost:
        return False, f"Estimated cost ${plan.estimated_total_cost_usd:.0f} exceeds the ${max_cost:.0f} budget."
    return True, f"{TRIP_DAYS}-day itinerary within the ${max_cost:.0f} budget."


def run_with_validated_retry(run_once, goal: str, max_attempts: int = 3) -> VacationPlan:
    """Hand-rolled retry: call `run_once(goal)`, check it, retry on failure."""
    for attempt in range(1, max_attempts + 1):
        candidate = run_once(goal)
        ok, reason = budget_check(candidate)
        print(f"  Attempt {attempt}: {'PASS' if ok else 'FAIL'} - {reason}")
        if ok:
            return candidate
    raise RuntimeError(f"No valid plan after {max_attempts} attempts.")


plan_within_budget = VacationPlan(
    destination="Lisbon, Portugal",
    itinerary=[DayPlan(day=day, activities=["Explore the old town"]) for day in range(1, TRIP_DAYS + 1)],
    estimated_total_cost_usd=400,
)
plan_missing_a_day = plan_within_budget.model_copy(update={"itinerary": plan_within_budget.itinerary[:-1]})

for label, plan in [("Plan within budget", plan_within_budget), ("Plan missing a day", plan_missing_a_day)]:
    ok, reason = budget_check(plan)
    print(f"{label}: {'PASS' if ok else 'FAIL'} - {reason}")

## Step 7. Set up the LangChain agent

- `ChatOllama` connects LangChain to the model. `num_predict` is its output cap.
- `reasoning=False` turns off thinking. `granite4.2:3b` thinks by default, and that hidden reasoning can use up the whole cap and leave an empty answer.
- `create_agent` builds the loop. `system_prompt` tells the model to compare a few destinations before deciding.
- `ToolCallLimitMiddleware` caps tool calls, and `exit_behavior="end"` stops quietly at the cap instead of raising an error.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL_TAG, num_predict=MAX_NEW_TOKENS, reasoning=False)

langchain_agent = create_agent(
    model=llm,
    tools=[destination_guide],
    system_prompt=(
        "You are a vacation-planning assistant. Use the destination_guide tool to "
        "check a couple of candidate destinations against the traveler's budget, "
        "season, and interests before deciding. Recommend exactly one destination "
        "and produce a day-by-day itinerary with a realistic total cost estimate."
    ),
    middleware=[ToolCallLimitMiddleware(run_limit=TOOL_CALL_BUDGET, exit_behavior="end")],
)

print("LangChain agent graph nodes:", list(langchain_agent.get_graph().nodes))
# Quick test call: the model should answer with no reasoning trace.
reply = llm.invoke("Reply with the single word OK.")
print("Model replied:", reply.content)
print("Reasoning trace length:", len(reply.additional_kwargs.get("reasoning_content") or ""), "characters (0 means thinking is off)")

## Step 8. Run the LangChain agent, type its output, validate it

- `create_agent`'s `response_format=` needs forced tool choice, which [Ollama doesn't support](https://reference.langchain.com/python/langchain-ollama/chat_models/ChatOllama/bind_tools). So typing takes two calls. The loop returns a plain-text plan, and `with_structured_output(VacationPlan)` turns it into a `VacationPlan`.
- A typed plan can still break the budget. `run_with_validated_retry` (Step 6) checks it and retries blind on failure.

In [ ]:
structured_llm = llm.with_structured_output(VacationPlan)


def run_langchain_agent(goal: str) -> VacationPlan:
    result = langchain_agent.invoke({"messages": [{"role": "user", "content": goal}]})
    final_text = result["messages"][-1].content  # the agent's last message: the plan as plain text
    print(f"  Messages exchanged in the tool-calling loop: {len(result['messages'])}")
    return structured_llm.invoke(
        f"Extract the destination, day-by-day itinerary, and estimated total cost from "
        f"this vacation plan:\n\n{final_text}"
    )


print("=== LANGCHAIN AGENT ===")
langchain_plan = run_with_validated_retry(run_langchain_agent, vacation_request)
print(langchain_plan.model_dump_json(indent=2))

## Step 9. Reuse the same tool in Mellea, zero rewrite

`MelleaTool.from_langchain()` wraps the same `BaseTool` the LangChain agent used, without changing it.

*Mellea guides: [Tools and Agents: LangChain and smolagents interop](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#langchain-and-smolagents-interop), [LangChain: Using LangChain tools](https://github.com/generative-computing/mellea/blob/main/docs/docs/integrations/langchain.md#using-langchain-tools)*

In [ ]:
mellea_destination_tool: MelleaTool = MelleaTool.from_langchain(destination_guide)

# The wrapper runs the exact same destination_guide function the LangChain agent used.
wrapped_result = mellea_destination_tool.run(destination="Lisbon, Portugal")
print("Wrapped tool returned:", wrapped_result)
print("Same result as the original tool:", wrapped_result == destination_guide.invoke({"destination": "Lisbon, Portugal"}))

## Step 10. Set up the Mellea agent

- The session uses `ChatContext`, which keeps the turn history, because Step 12 reuses the agent's history.
- `ModelOption.THINKING: False` turns off thinking, like `reasoning=False` in Step 7. `granite4.2:3b` thinks by default, and that hidden reasoning can use up the whole cap and leave an empty answer.
- `react()` is Mellea's ReACT loop. It runs on the session's backend and records its turns in a fresh `ChatContext`. It ends when the model calls the built-in `final_answer` tool, or raises an error when `loop_budget` runs out.
- `format=VacationPlan` shapes the final answer into the schema in one call.
- `run_mellea_agent()` returns the plan and the loop's history. `react()` is async, so it's called with `await`.

*Note:* `react()`'s `result.parsed_repr` is plain text here, not a `VacationPlan`, although the [ReACT how-to](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#react-agent) reads it as one. So `run_mellea_agent()` parses `result.value` itself.

*Mellea guides: [Tools and Agents: ReACT agent](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/tools-and-agents.md#react-agent), [Context and Sessions: Context types](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/use-context-and-sessions.md#context-types), [Model Options: Reasoning and thinking mode](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/configure-model-options.md#reasoning-and-thinking-mode)*

In [ ]:
m: MelleaSession = start_session(
    backend_name="ollama",
    model_id=OLLAMA_MODEL_TAG,
    ctx=ChatContext(),
    model_options={
        ModelOption.MAX_NEW_TOKENS: MAX_NEW_TOKENS,
        ModelOption.THINKING: False,  # same as reasoning=False for ChatOllama in Step 7
    },
)


async def run_mellea_agent(goal: str) -> tuple[VacationPlan, ChatContext]:
    result, agent_ctx = await react(
        goal=goal,
        context=ChatContext(),
        backend=m.backend,
        tools=[mellea_destination_tool],
        format=VacationPlan,
        loop_budget=TOOL_CALL_BUDGET,  # same cap used for the LangChain agent above
    )
    # result.value is the plan as JSON text
    return VacationPlan.model_validate_json(result.value or ""), agent_ctx


# Quick test call on a copy of the session (m.clone()), so m's own history stays empty.
reply = m.clone().instruct("Reply with the single word OK.")
print("Model replied:", reply.value)
print("Thinking trace length:", len(reply.thinking or ""), "characters (0 means thinking is off)")

## Step 11. Run the Mellea agent

Mellea logs `## ReACT TURN NUMBER N` for each turn of the loop.

In [ ]:
mellea_plan, agent_ctx = await run_mellea_agent(vacation_request)

print("=== MELLEA AGENT PLAN ===")
print(mellea_plan.model_dump_json(indent=2))

## Step 12. A Requirement-checked, auto-repaired synthesis

`react()` takes no `requirements=`, so a `@generative` call that does writes the final plan, using the facts Step 11 gathered.

- `write_vacation_plan` is a `@generative` function. Its docstring is the instruction, `request` the input, and `-> VacationPlan` the output type. It returns a parsed `VacationPlan`.
- `budget_requirement` is a rule checked after each attempt. Its `validation_fn` runs `budget_check` and prints PASS or FAIL. The rule's text also goes into the prompt.
- `RepairTemplateStrategy` adds the failure reason to the next attempt's prompt. Step 8's retry has no equivalent.
- `m.ctx = agent_ctx` gives the session Step 11's tool results, so no second tool loop runs.

*Note:* A `@generative` call's raw output is wrapped as `{"result": {...}}`, so parsing `output.value` as a `VacationPlan` fails inside a validator. The [Requirements on `@generative` output](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/enforce-structured-output.md#requirements-on-generative-output) guide doesn't mention the wrapper. So `validate_budget` reads the already-parsed `parsed_repr`.

*Mellea guides: [Tutorial 04: Making agents reliable](https://github.com/generative-computing/mellea/blob/main/docs/docs/tutorials/04-making-agents-reliable.md), [Generative Functions: Basic `@generative`](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/generative-functions.md#basic-generative), [Enforce Structured Output: Requirements on `@generative` output](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/enforce-structured-output.md#requirements-on-generative-output), [Write Custom Verifiers: Writing a full validation function](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/write-custom-verifiers.md#writing-a-full-validation-function), [Instruct-Validate-Repair: Sampling strategies and the IVR loop](https://github.com/generative-computing/mellea/blob/main/docs/docs/concepts/instruct-validate-repair.md#sampling-strategies-and-the-ivr-loop)*

In [ ]:
@generative
def write_vacation_plan(request: str) -> VacationPlan:
    """Using the destination facts already gathered, write the final vacation plan for this request."""


def validate_budget(ctx) -> ValidationResult:
    """Requirement validator: reuse budget_check and print the verdict for each attempt."""
    plan = ctx.last_output().parsed_repr  # @generative already parsed it into a VacationPlan
    ok, reason = budget_check(plan)
    print(f"  {'PASS' if ok else 'FAIL'} - {reason}")
    return ValidationResult(ok, reason=None if ok else reason)


budget_requirement = Requirement(
    "The itinerary must cover every requested day, and the estimated total cost "
    "must not exceed the traveler's budget.",
    validation_fn=validate_budget,
)

m.ctx = agent_ctx
print("=== VALIDATING MELLEA AGENT ===")
validated_mellea_plan = write_vacation_plan(
    m,
    request=vacation_request,
    requirements=[budget_requirement],
    strategy=RepairTemplateStrategy(loop_budget=3),
)
print(validated_mellea_plan.model_dump_json(indent=2))

### Side-by-side comparison

<table>
  <tr>
    <th width="16%"></th>
    <th width="42%">LangChain Agent</th>
    <th width="42%">Mellea Agent</th>
  </tr>
  <tr>
    <td><strong>Setup</strong></td>
    <td><code>ChatOllama</code> + <code>create_agent(model, tools, system_prompt, middleware)</code></td>
    <td><code>start_session()</code> + <code>react(goal, context, backend, tools, format, loop_budget)</code></td>
  </tr>
  <tr>
    <td><strong>Tool definition</strong></td>
    <td><code>destination_guide</code> decorated with <code>@tool</code></td>
    <td>Same object, wrapped with <code>MelleaTool.from_langchain()</code>, zero rewrite</td>
  </tr>
  <tr>
    <td><strong>Output typing</strong></td>
    <td>Needs the provider to support forced tool choice; <code>ChatOllama</code> doesn't, so a local model needs a second <code>with_structured_output()</code> call</td>
    <td><code>format=VacationPlan</code> on <code>react()</code> itself, using native constrained decoding with no tool-choice dependency</td>
  </tr>
  <tr>
    <td><strong>Typed function</strong></td>
    <td>No equivalent. Prompts are strings, and typing is a separate call</td>
    <td>A <code>@generative</code> function whose signature and docstring become the call. It returns a parsed <code>VacationPlan</code> and takes <code>requirements=</code> / <code>strategy=</code> (Step 12)</td>
  </tr>
  <tr>
    <td><strong>Loop control</strong></td>
    <td><code>ToolCallLimitMiddleware(run_limit=...)</code>, one option from a larger middleware catalog</td>
    <td><code>loop_budget=...</code>, a plain keyword argument</td>
  </tr>
  <tr>
    <td><strong>Validation</strong></td>
    <td>Hand-written retry wrapper; retries blind, same prompt every time</td>
    <td><code>Requirement</code> + <code>RepairTemplateStrategy</code>, which feeds the failure <em>reason</em> back into the next attempt</td>
  </tr>
  <tr>
    <td><strong>Observability</strong></td>
    <td><code>debug=True</code>, or <code>.stream(..., stream_mode="updates")</code>; message count printed in Step 8</td>
    <td>Per-turn <code>## ReACT TURN NUMBER N</code> logging by default; <code>ChatContext</code> is directly inspectable</td>
  </tr>
</table>

## Key Takeaways

- A typed schema guarantees *shape*, not *content*. Both agents still needed `budget_check`.
- With a thinking model, turn thinking off and leave the token cap room for the largest output, here `final_answer`.
- `Requirement` + repair strategies work on every `instruct()` and `@generative` call, not just agents. See [Mellea vs Orchestration Frameworks](https://github.com/generative-computing/mellea/blob/main/docs/docs/concepts/architecture-vs-agents.md).

## Next Steps

- [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb): more on `Requirement` and the retry and repair loops from Steps 8 and 12.
- [Structured Data Extraction](../StructuredDataExtraction/StructuredDataExtraction.ipynb): more on `format=` and Pydantic schemas.
- [Structured Sentiment Classifier](../SentimentClassifier/SentimentClassifier.ipynb): the same raw-call-vs-Mellea contrast, on a single classification call.